In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sbhatti/financial-sentiment-analysis")

print("Path to dataset files:", path)

import pandas as pd

data = pd.read_csv(path+"\data.csv")
data.head()

Path to dataset files: C:\Users\gael1\.cache\kagglehub\datasets\sbhatti\financial-sentiment-analysis\versions\4


,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral


In [10]:
df=data.copy()
df

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral
...,...,...
5837,RISING costs have forced packaging producer Hu...,negative
5838,Nordic Walking was first used as a summer trai...,neutral
5839,"According shipping company Viking Line , the E...",neutral
5840,"In the building and home improvement trade , s...",neutral


In [ ]:
features= df.drop(['Sentiment'], axis=1)
features

,Sentence
0,The GeoSolutions technology will leverage Bene...
1,"$ESI on lows, down $1.50 to $2.50 BK a real po..."
2,"For the last quarter of 2010 , Componenta 's n..."
3,According to the Finnish-Russian Chamber of Co...
4,The Swedish buyout firm has sold its remaining...
...,...
5837,RISING costs have forced packaging producer Hu...
5838,Nordic Walking was first used as a summer trai...
5839,"According shipping company Viking Line , the E..."
5840,"In the building and home improvement trade , s..."


In [12]:
labels= df['Sentiment']
labels

0       positive
1       negative
2       positive
3        neutral
4        neutral
          ...   
5837    negative
5838     neutral
5839     neutral
5840     neutral
5841    positive
Name: Sentiment, Length: 5842, dtype: object

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)
# Juntamos X_train y y_train temporalmente
train_df = X_train.copy()
train_df["label"] = y_train.values

# Número de muestras de la clase minoritaria
n_samples = train_df["label"].value_counts().min()

# Tomamos la misma cantidad de cada clase
train_balanced = (
    train_df.groupby("label", group_keys=False)
    .sample(n=n_samples, random_state=42)
)

# Separar nuevamente X e y
X_train = train_balanced.drop(columns="label")
y_train = train_balanced["label"]

print(y_train.value_counts())

label
negative    688
neutral     688
positive    688
Name: count, dtype: int64


In [12]:
# train_model.py
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

def deploy_model(train_X, train_y):
    vectorizer = TfidfVectorizer()
    X_vec = vectorizer.fit_transform(train_X)

    model = LogisticRegression()
    model.fit(X_vec, train_y)

    with open("model.pkl", "wb") as f:
        pickle.dump(model, f)

    with open("vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)

In [20]:
model = deploy_model(X_train["Sentence"], y_train)
model

In [ ]:

import torch
print(torch.__version__)



In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

model_name = "prajjwal1/bert-tiny"

tokenizer = BertTokenizer.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

c:\Users\gael1\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gael1\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 17.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.7MB            

model.safetensors: downloading bytes:           |  0.00B            

In [6]:

def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    return label_map[prediction]

In [42]:

y_train_num = y_train.map(label_map).to_list()
y_test_num = y_test.map(label_map).to_list()

In [ ]:
example = X_train["Sentence"].iloc[0]

print(example)

tokens = tokenizer.tokenize(example)
print(tokens)

encoded = tokenizer(example)
print(encoded["input_ids"])

In [34]:
train_df = X_train["Sentence"].astype(str).to_list()

test_df = X_test["Sentence"].astype(str).to_list()

In [44]:
train_encodings = tokenizer(
    train_df,
    padding=True,
    truncation=True,
    max_length=128
)

test_encodings = tokenizer(
    test_df,
    padding=True,
    truncation=True,
    max_length=128
)

In [45]:
import torch

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [46]:
train_dataset = SentimentDataset(
    train_encodings,
    y_train_num
)

test_dataset = SentimentDataset(
    test_encodings,
    y_test_num
)

In [47]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.069214,1.042387
2,0.995732,0.980051
3,0.966362,0.974039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\gael1\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\gael1\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=387, training_loss=1.0188814700419897, metrics={'train_runtime': 45.0544, 'train_samples_per_second': 137.434, 'train_steps_per_second': 8.59, 'total_flos': 1429386308064.0, 'train_loss': 1.0188814700419897, 'epoch': 3.0})

In [48]:
model.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sentiment_model\\tokenizer_config.json',
 './sentiment_model\\tokenizer.json')